# geeViz MCP Server Test Suite

Tests the 30 MCP tools by connecting a Gemini model to the live geeViz MCP server.
Each section targets a tool category with focused prompts.

**Architecture:** A single MCP server subprocess is started once and reused across all tests
(avoids re-spawning and re-initializing Earth Engine for every cell). Timeouts prevent hangs.

**Setup:**
1. Create a `.env` file in the project root with `GOOGLE_API_KEY=your-key-here`
2. `pip install google-genai mcp python-dotenv`

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
import subprocess
import sys
import time
from contextlib import AsyncExitStack

import dotenv
dotenv.load_dotenv()

# Path setup -- ensure geeViz package is importable
_THIS_DIR = os.path.dirname(os.path.abspath("__file__"))
_PACKAGE_ROOT = os.path.abspath(os.path.join(_THIS_DIR, "..", ".."))
if _PACKAGE_ROOT not in sys.path:
    sys.path.insert(0, _PACKAGE_ROOT)

from google import genai
from google.genai import types as gtypes
from mcp.client.session import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client

# Config
MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
API_KEY = os.environ.get("GOOGLE_API_KEY")
if not API_KEY:
    raise RuntimeError("GOOGLE_API_KEY not found. Add it to a .env file in the project root.")

MAX_TOOL_ROUNDS = 15
RESULT_TRUNCATE = 30_000
GEMINI_TIMEOUT = 120       # seconds per Gemini API call
MCP_TOOL_TIMEOUT = 180     # seconds per MCP tool call
TEST_TIMEOUT = 600         # seconds total per test

gemini = genai.Client(api_key=API_KEY)
print(f"Model: {MODEL}")
print("Gemini client ready.")

## Helper Functions

Schema conversion (MCP -> Gemini), persistent server session, and the test runner.

In [ ]:
_JSON_TO_GEMINI_TYPE = {
    "string": "STRING",
    "integer": "INTEGER",
    "number": "NUMBER",
    "boolean": "BOOLEAN",
    "array": "ARRAY",
    "object": "OBJECT",
}


def _schema_to_gemini(json_schema: dict) -> gtypes.Schema:
    """Convert a JSON Schema dict to a Gemini Schema."""
    gtype = _JSON_TO_GEMINI_TYPE.get(json_schema.get("type", "string"), "STRING")
    props = {}
    for pname, pschema in json_schema.get("properties", {}).items():
        props[pname] = gtypes.Schema(
            type=_JSON_TO_GEMINI_TYPE.get(pschema.get("type", "string"), "STRING"),
            description=pschema.get("description", ""),
        )
    required = json_schema.get("required", [])
    return gtypes.Schema(type=gtype, properties=props, required=required or None)


def _mcp_tools_to_gemini(mcp_tools) -> gtypes.Tool:
    """Convert MCP Tool objects to a Gemini Tool with FunctionDeclarations."""
    decls = []
    for tool in mcp_tools:
        schema = _schema_to_gemini(tool.inputSchema) if tool.inputSchema else None
        decls.append(gtypes.FunctionDeclaration(
            name=tool.name,
            description=tool.description or "",
            parameters=schema,
        ))
    return gtypes.Tool(function_declarations=decls)


def _short_args(args: dict, maxlen: int = 120) -> str:
    s = ", ".join(f"{k}={repr(v)[:60]}" for k, v in args.items())
    return s[:maxlen] + ("..." if len(s) > maxlen else "")

In [ ]:
# ---------------------------------------------------------------------------
# Persistent MCP server session -- started once, reused across all tests
# ---------------------------------------------------------------------------
_exit_stack: AsyncExitStack | None = None
_mcp_session: ClientSession | None = None
_gemini_tools: gtypes.Tool | None = None


async def start_mcp_server():
    """Start the MCP server subprocess and keep the session open."""
    global _exit_stack, _mcp_session, _gemini_tools

    if _mcp_session is not None:
        print("MCP server already running.")
        return

    server_params = StdioServerParameters(
        command=sys.executable,
        args=["-m", "geeViz.mcp.server"],
        cwd=_PACKAGE_ROOT,
    )

    _exit_stack = AsyncExitStack()
    read_stream, write_stream = await _exit_stack.enter_async_context(
        stdio_client(server_params, errlog=subprocess.DEVNULL)
    )
    _mcp_session = await _exit_stack.enter_async_context(
        ClientSession(read_stream, write_stream)
    )
    await _mcp_session.initialize()

    tools_result = await _mcp_session.list_tools()
    _gemini_tools = _mcp_tools_to_gemini(tools_result.tools)
    tool_names = [t.name for t in tools_result.tools]
    print(f"MCP server started: {len(tool_names)} tools available")
    print(f"Tools: {', '.join(tool_names)}")


async def stop_mcp_server():
    """Shut down the MCP server subprocess."""
    global _exit_stack, _mcp_session, _gemini_tools
    if _exit_stack:
        await _exit_stack.aclose()
    _exit_stack = None
    _mcp_session = None
    _gemini_tools = None
    print("MCP server stopped.")


async def run_mcp_test(
    prompt: str,
    max_rounds: int = MAX_TOOL_ROUNDS,
    verbose: bool = True,
    test_timeout: int = TEST_TIMEOUT,
) -> dict:
    """Run a prompt through Gemini + the persistent MCP server session.

    Returns dict with keys: response, tools_used, rounds, elapsed_seconds, error
    """
    if _mcp_session is None or _gemini_tools is None:
        return {"response": "", "tools_used": [], "rounds": 0, "elapsed_seconds": 0,
                "error": "MCP server not started. Run `await start_mcp_server()` first."}

    async def _inner():
        t0 = time.time()
        tools_used = []
        history = [gtypes.Content(role="user", parts=[gtypes.Part(text=prompt)])]

        for round_num in range(1, max_rounds + 1):
            # Gemini API call with timeout
            try:
                resp = await asyncio.wait_for(
                    asyncio.to_thread(
                        gemini.models.generate_content,
                        model=MODEL,
                        contents=history,
                        config=gtypes.GenerateContentConfig(tools=[_gemini_tools]),
                    ),
                    timeout=GEMINI_TIMEOUT,
                )
            except asyncio.TimeoutError:
                msg = f"Gemini API timed out after {GEMINI_TIMEOUT}s on round {round_num}"
                if verbose:
                    print(f"  TIMEOUT: {msg}")
                return {"response": msg, "tools_used": tools_used,
                        "rounds": round_num, "elapsed_seconds": round(time.time() - t0, 1),
                        "error": "gemini_timeout"}

            fn_calls = [
                p for p in (resp.candidates[0].content.parts or [])
                if p.function_call
            ]

            if not fn_calls:
                text = resp.text or "(empty response)"
                if verbose:
                    print(f"  Done in {round_num - 1} tool rounds ({round(time.time() - t0, 1)}s)")
                return {"response": text, "tools_used": tools_used,
                        "rounds": round_num - 1, "elapsed_seconds": round(time.time() - t0, 1)}

            history.append(resp.candidates[0].content)

            response_parts = []
            for part in fn_calls:
                fc = part.function_call
                fn_name = fc.name
                fn_args = dict(fc.args) if fc.args else {}
                tools_used.append(fn_name)

                if verbose:
                    print(f"  [round {round_num}] {fn_name}({_short_args(fn_args)})")

                # MCP tool call with timeout
                try:
                    mcp_result = await asyncio.wait_for(
                        _mcp_session.call_tool(name=fn_name, arguments=fn_args),
                        timeout=MCP_TOOL_TIMEOUT,
                    )
                    result_str = "\n".join(
                        c.text for c in mcp_result.content if hasattr(c, "text")
                    )
                    if mcp_result.isError:
                        result_str = json.dumps({"error": result_str})
                except asyncio.TimeoutError:
                    result_str = json.dumps({"error": f"MCP tool '{fn_name}' timed out after {MCP_TOOL_TIMEOUT}s"})
                    if verbose:
                        print(f"  TIMEOUT: {fn_name}")
                except Exception as exc:
                    result_str = json.dumps({"error": str(exc)})

                if len(result_str) > RESULT_TRUNCATE:
                    result_str = result_str[:RESULT_TRUNCATE] + "\n...(truncated)"

                response_parts.append(
                    gtypes.Part(function_response=gtypes.FunctionResponse(
                        name=fn_name, response={"result": result_str}
                    ))
                )

            history.append(gtypes.Content(role="user", parts=response_parts))

        if verbose:
            print(f"  Hit max rounds ({max_rounds})")
        return {"response": "(max rounds reached)", "tools_used": tools_used,
                "rounds": max_rounds, "elapsed_seconds": round(time.time() - t0, 1)}

    # Wrap entire test in a timeout
    try:
        return await asyncio.wait_for(_inner(), timeout=test_timeout)
    except asyncio.TimeoutError:
        return {"response": "", "tools_used": [], "rounds": 0,
                "elapsed_seconds": test_timeout,
                "error": f"Entire test timed out after {test_timeout}s"}

## Start MCP Server

Run this once. The server stays alive for all subsequent test cells.
Re-run to restart if the server dies.

In [ ]:
await start_mcp_server()

## Test 1: Environment & Version Info

Tests: `get_version_info`, `get_project_info`, `get_namespace`

In [ ]:
result = await run_mcp_test(
    "What versions of geeViz, Earth Engine, and Python are installed? "
    "What EE project is active? What variables exist in the REPL namespace?"
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:2000]}")

## Test 2: API Introspection

Tests: `search_functions`, `get_api_reference`, `get_reference_data`

In [ ]:
result = await run_mcp_test(
    "I want to do change detection. Search for LANDTRENDR-related functions across all modules, "
    "then get the full API reference for the simpleLANDTRENDR function. "
    "Also, list all functions in the changeDetectionLib module."
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:3000]}")

## Test 3: Dataset Discovery & Asset Inspection

Tests: `search_datasets`, `get_catalog_info`, `inspect_asset`

In [ ]:
result = await run_mcp_test(
    "Search the GEE catalog for 'NLCD land cover'. Get detailed catalog info for the top result. "
    "Then inspect the NLCD asset to see its real bands, date range, and properties. "
    "Filter to images from 2019-2021."
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:3000]}")

## Test 4: Code Execution & Map Visualization

Tests: `run_code`, `view_map`, `get_map_layers`, `clear_map`

In [ ]:
result = await run_mcp_test(
    "Clear the map, then write code to load a Sentinel-2 composite for summer 2023 "
    "near Yellowstone National Park using geeViz wrappers (getSentinel2Wrapper). "
    "Add a true-color layer and an NDVI layer to the map. Then open the map and tell me what layers are on it."
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:3000]}")

## Test 5: Example Discovery

Tests: `list_examples`, `get_example`

In [ ]:
result = await run_mcp_test(
    "List available geeViz examples that relate to LANDTRENDR or change detection. "
    "Then read the full source of the LANDTRENDRWrapper example and summarize what it does."
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:3000]}")

## Test 6: Zonal Summary & Charting

Tests: `extract_and_chart` (bar chart from single image)

In [ ]:
result = await run_mcp_test(
    "Use extract_and_chart to create a bar chart of NLCD 2021 land cover classes "
    "within a 10km buffer around Bozeman, Montana. Show me the data table and chart."
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:3000]}")

## Test 7: Time Series Charting

Tests: `extract_and_chart` (time series from ImageCollection), `run_code`

In [ ]:
result = await run_mcp_test(
    "Create an NDVI time series chart for Landsat data from 2018-2024 "
    "at a point in Yellowstone (lon=-110.5, lat=44.6). "
    "Use geeViz getLandsatWrapper to get annual composites, then use extract_and_chart "
    "to chart the NDVI trend. Show the data table and chart."
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:3000]}")

## Test 8: Geocoding

Tests: `geocode`

In [ ]:
result = await run_mcp_test(
    "Geocode 'Grand Teton National Park' and 'Glacier National Park'. "
    "Return the coordinates and any GEE boundary asset IDs."
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:2000]}")

## Test 9: Session Save

Tests: `save_session` (both formats)

In [ ]:
result = await run_mcp_test(
    "Run some simple code: print('hello from MCP test'). "
    "Then save the session as a Python script and also as a Jupyter notebook. "
    "Tell me the file paths for both."
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:2000]}")

## Test 10: Asset Management

Tests: `list_assets`, `track_tasks`

In [ ]:
result = await run_mcp_test(
    "List the top-level assets in my Earth Engine project. "
    "Also check if there are any running or recent EE tasks."
)
print(f"\nTools used: {result['tools_used']}")
print(f"Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:2000]}")

## Test 11: End-to-End Workflow

Tests multiple tools together: a realistic multi-step analysis that should exercise
`search_functions`, `get_api_reference`, `run_code`, `extract_and_chart`, and `view_map`.

In [ ]:
result = await run_mcp_test(
    "Do a complete analysis: "
    "1. Find the geeViz function for getting Landsat data (search for 'getLandsatWrapper'). "
    "2. Get its API reference to check the signature. "
    "3. Write and run code to get Landsat annual composites for 2020-2024 near Bozeman, MT. "
    "4. Add NDVI layers to the map. "
    "5. Use extract_and_chart to chart NDVI trends at the center point. "
    "6. Open the map. "
    "Show me the NDVI data table, chart, and map URL.",
    max_rounds=20,
)
print(f"\nTools used: {result['tools_used']}")
print(f"Rounds: {result['rounds']}, Elapsed: {result['elapsed_seconds']}s")
if result.get("error"):
    print(f"ERROR: {result['error']}")
print(f"\n{result['response'][:4000]}")

## Cleanup

Stop the MCP server subprocess when done.

In [ ]:
await stop_mcp_server()